In [8]:
# Bloco 1 - Importar a base de veículos com cidade e estado

from pathlib import Path
import pandas as pd


def obter_pasta_do_notebook():
    return Path.cwd()


def importar_csv_mesma_pasta(nome_arquivo, nome_base, colunas=None, sep=",", engine=None):
    try:
        pasta_notebook = obter_pasta_do_notebook()
        caminho_arquivo = pasta_notebook / nome_arquivo

        argumentos = {
            "filepath_or_buffer": caminho_arquivo,
            "sep": sep,
            "encoding": "utf-8-sig"
        }

        if engine is not None:
            argumentos["engine"] = engine
        else:
            argumentos["low_memory"] = False

        alias_map = {
            "manutenc": "manutencao",
            "id_firebas": "id_firebase",
            "seguro_ar": "seguro_anual",
            "tipo_comb": "tipo_combustivel",
            "numero_d": "numero_de_pneus",
            "vida_util": "vida_util_do_pneu",
            "potencia": "potencia_do_motor",
            "outro_tipc": "outro_tipo_text"
        }

        if colunas is not None:
            try:
                header_kwargs = {
                    "filepath_or_buffer": caminho_arquivo,
                    "sep": sep,
                    "encoding": "utf-8-sig",
                    "nrows": 0
                }

                if engine is not None:
                    header_kwargs["engine"] = engine
                    if engine != "python":
                        header_kwargs["low_memory"] = False
                else:
                    header_kwargs["low_memory"] = False

                df_header = pd.read_csv(**header_kwargs)
                df_header.columns = df_header.columns.str.strip()

                normalized_columns = {
                    col.strip().lower(): col.strip()
                    for col in df_header.columns
                }

                selected_cols = []
                missing_cols = []

                for desejada in colunas:
                    chave = desejada.strip().lower()
                    if chave in normalized_columns:
                        selected_cols.append(normalized_columns[chave])
                    elif chave in alias_map and alias_map[chave].lower() in normalized_columns:
                        selected_cols.append(normalized_columns[alias_map[chave].lower()])
                    else:
                        missing_cols.append(desejada)

                if missing_cols:
                    print(f"Atenção: algumas colunas não foram encontradas em '{nome_arquivo}':")
                    print(missing_cols)
                    if selected_cols:
                        print("Serão lidas apenas as colunas disponíveis.")
                    else:
                        print("Nenhuma das colunas solicitadas foi encontrada. Lendo todas as colunas.")

                if selected_cols:
                    argumentos["usecols"] = selected_cols
            except Exception as erro_header:
                print(f"Aviso: não foi possível validar colunas do arquivo '{nome_arquivo}'.")
                print(f"Detalhes: {erro_header}")
                argumentos["usecols"] = colunas

        df = pd.read_csv(**argumentos)
        df.columns = df.columns.str.strip()

        available_columns = {
            col.strip().lower(): col.strip()
            for col in df.columns
        }
        rename_map = {}

        for canonical, actual in alias_map.items():
            if actual.lower() in available_columns:
                rename_map[available_columns[actual.lower()]] = canonical

        if rename_map:
            df = df.rename(columns=rename_map)

        print(f"Base '{nome_base}' importada com sucesso!")
        print(f"Arquivo importado: {caminho_arquivo}")
        print(f"Quantidade de linhas: {df.shape[0]}")
        print(f"Quantidade de colunas: {df.shape[1]}")
        print()

        return df

    except FileNotFoundError:
        print(f"Erro: arquivo da base '{nome_base}' não encontrado.")
        print(f"Verifique se o arquivo '{nome_arquivo}' está na mesma pasta do notebook.")
        return None

    except Exception as erro:
        print(f"Erro ao importar a base '{nome_base}'.")
        print(f"Detalhes do erro: {erro}")
        return None


def inserir_cidade_estado(df_veiculos, df_municipios):
    df_veiculos = df_veiculos.copy()
    df_municipios = df_municipios.copy()

    df_veiculos.columns = df_veiculos.columns.str.strip()
    df_municipios.columns = df_municipios.columns.str.strip()

    colunas_municipios_necessarias = [
        "Código Município Completo",
        "Nome_Município",
        "Nome_UF"
    ]

    colunas_faltantes = [
        coluna
        for coluna in colunas_municipios_necessarias
        if coluna not in df_municipios.columns
    ]

    if colunas_faltantes:
        print("Erro nas colunas da base de municípios.")
        print(colunas_faltantes)
        return None

    df_veiculos["codigo_cidade"] = pd.to_numeric(
        df_veiculos["codigo_cidade"],
        errors="coerce"
    ).astype("Int64")

    df_municipios["Código Município Completo"] = pd.to_numeric(
        df_municipios["Código Município Completo"],
        errors="coerce"
    ).astype("Int64")

    municipios = (
        df_municipios[
            [
                "Código Município Completo",
                "Nome_Município",
                "Nome_UF"
            ]
        ]
        .drop_duplicates(subset=["Código Município Completo"])
        .rename(
            columns={
                "Código Município Completo": "codigo_cidade",
                "Nome_Município": "nome_cidade",
                "Nome_UF": "estado"
            }
        )
    )

    df_veiculos = df_veiculos.merge(
        municipios,
        on="codigo_cidade",
        how="left"
    )

    colunas = list(df_veiculos.columns)

    if "nome_cidade" in colunas:
        colunas.remove("nome_cidade")
    if "estado" in colunas:
        colunas.remove("estado")

    if "codigo_cidade" in colunas:
        posicao = colunas.index("codigo_cidade")
        colunas.insert(posicao + 1, "nome_cidade")
        colunas.insert(posicao + 2, "estado")

    df_veiculos = df_veiculos[colunas]

    print("Cidade e estado inseridos com sucesso!")
    print(
        "Registros sem município:",
        df_veiculos["nome_cidade"].isna().sum()
    )

    return df_veiculos


nome_arquivo_veiculos = "script_04_veiculos.csv"
nome_arquivo_municipios = "Lista de Municipios IBGE.csv"

colunas_veiculos = [
    "codigo_cidade",
    "id_veiculo",
    "placa",
    "ano",
    "modo",
    "origem",
    "km_inicial",
    "capacidade",
    "km_atual",
    "tipo",
    "renavam",
    "manutenc",
    "marca",
    "id_firebas",
    "ipva",
    "dpvat",
    "seguro_ar",
    "consumo",
    "tipo_comb",
    "numero_d",
    "vida_util",
    "preco",
    "potencia",
    "modelo",
    "outro_tipc",
    "is_state",
    "historico",
    "ativo",
    "dt_criacao",
    "criado_id",
    "dt_alteracao",
    "alterado_id"
]

df_veiculos = importar_csv_mesma_pasta(
    nome_arquivo_veiculos,
    "veículos",
    colunas=colunas_veiculos,
    sep=",")

df_municipios = importar_csv_mesma_pasta(
    nome_arquivo_municipios,
    "municípios",
    sep=None,
    engine="python"
)

if df_veiculos is not None and df_municipios is not None:
    df_veiculos_transformado = inserir_cidade_estado(
        df_veiculos,
        df_municipios
    )

    if df_veiculos_transformado is not None:
        display(df_veiculos_transformado)

        pasta_notebook = obter_pasta_do_notebook()
        nome_arquivo_saida = "script_04_veiculos_consolidado.csv"
        caminho_arquivo_saida = pasta_notebook / nome_arquivo_saida

        df_veiculos_transformado.to_csv(
            caminho_arquivo_saida,
            index=False,
            encoding="utf-8-sig"
        )

        print("Arquivo exportado com sucesso!")
        print(f"Caminho: {caminho_arquivo_saida}")


Base 'veículos' importada com sucesso!
Arquivo importado: c:\Users\Matheus Henrique\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_04_veiculos.csv
Quantidade de linhas: 67730
Quantidade de colunas: 32

Base 'municípios' importada com sucesso!
Arquivo importado: c:\Users\Matheus Henrique\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Lista de Municipios IBGE.csv
Quantidade de linhas: 5572
Quantidade de colunas: 13

Cidade e estado inseridos com sucesso!
Registros sem município: 0


,codigo_cidade,nome_cidade,estado,id_veiculo,placa,ano,modo,origem,km_inicial,capacidade,...,potencia,modelo,outro_tipc,is_state,historico,ativo,dt_criacao,criado_id,dt_alteracao,alterado_id
0,3201159,Brejetuba,Espírito Santo,63203,MSB-5F90,1999,0,2,0.00,24,...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
1,1100031,Cabixi,Rondônia,0,OHN-8878,2012,0,1,0.00,14,...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
2,1100056,Cerejeiras,Rondônia,0,NEG-2868,2014,0,1,0.00,48,...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
3,1100080,Costa Marques,Rondônia,0,NEF-9378,2014,0,1,1839.42,57,...,NaN,4,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
4,1100098,Espigão D'Oeste,Rondônia,0,NBW-0502,2013,0,1,0.00,48,...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67725,4100202,Adrianópolis,Paraná,68348,AXX-8I31,2022,0,2,0.00,24,...,115.0,0,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
67726,3128808,Guidoval,Minas Gerais,69159,TDR-7J84,2024,0,1,0.00,31,...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
67727,2926608,Ribeira do Pombal,Bahia,69862,OLA-8139,2013,0,1,0.00,48,...,186.0,4,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
67728,2312403,São Gonçalo do Amarante,Ceará,42118,NUZ-0152,2010,0,2,0.00,70,...,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


Arquivo exportado com sucesso!
Caminho: c:\Users\Matheus Henrique\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_04_veiculos_consolidado.csv


In [9]:
# Bloco 2 - Preparar a base de veículos para consolidação

df_consolidacao = df_veiculos_transformado.copy()

df_consolidacao["estado"] = (
    df_consolidacao["estado"]
    .fillna("Não identificado")
)

colunas_inteiras = [
    "id_veiculo",
    "ano",
    "modo",
    "origem",
    "tipo",
    "marca"
]

colunas_float = [
    "km_inicial",
    "capacidade",
    "km_atual"
]

for coluna in colunas_inteiras:
    if coluna in df_consolidacao.columns:
        df_consolidacao[coluna] = pd.to_numeric(
            df_consolidacao[coluna],
            errors="coerce"
        ).astype("Int64")

for coluna in colunas_float:
    if coluna in df_consolidacao.columns:
        df_consolidacao[coluna] = pd.to_numeric(
            df_consolidacao[coluna],
            errors="coerce"
        )

colunas_categoricas = [
    "tipo_comb",
    "modelo",
    "outro_tipc",
    "seguro_ar"
]

for coluna in colunas_categoricas:
    if coluna in df_consolidacao.columns:
        df_consolidacao[coluna] = (
            df_consolidacao[coluna]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.upper()
        )

modo_map = {
    0: "Rodoviário",
    1: "Aquaviário"
}

origem_map = {
    1: "Veículo Próprio",
    2: "Veículo Terceirizado"
}

tipo_map = {
    1: "Ônibus",
    2: "Micro-ônibus",
    3: "Van",
    4: "Kombi",
    5: "Caminhão",
    6: "Caminhonete",
    7: "Motocicleta",
    8: "Animal de tração",
    9: "Lancha/Voadeira",
    10: "Barco de Madeira",
    11: "Barco de alumínio",
    12: "Canoa motorizada",
    13: "Canoa a remo",
    14: "Bicicleta",
    99: "Outro"
}

marca_map = {
    1: "Iveco",
    2: "Mercedes-Benz",
    3: "Renault",
    4: "Volkswagen",
    5: "Volare",
    6: "EMGEPRON",
    7: "ESTALEIRO B3",
    8: "Yamaha",
    10: "Outra",
    11: "SCANIA",
    12: "MAN",
    13: "CHEVROLET",
    14: "FIAT",
    15: "FORD",
    16: "HYUNDAI",
    17: "KIA",
    18: "TOYOTA",
    19: "AGRALE",
    20: "VOLVO"
}

comb_map = {
    "G": "Gasolina",
    "D": "Diesel",
    "E": "Etanol",
    "N": "Gás Natural",
    "O": "Outro",
    "": "Não informado"
}

for coluna, mapa in [
    ("modo", modo_map),
    ("origem", origem_map),
    ("tipo", tipo_map),
    ("marca", marca_map)
]:
    if coluna in df_consolidacao.columns:
        df_consolidacao[f"{coluna}_desc"] = (
            df_consolidacao[coluna]
            .map(mapa)
            .fillna("Não informado")
        )

if "tipo_comb" in df_consolidacao.columns:
    df_consolidacao["tipo_combustivel"] = (
        df_consolidacao["tipo_comb"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .map(comb_map)
        .fillna("Não informado")
    )

print("Base de veículos preparada para consolidação.")

display(df_consolidacao.head())


Base de veículos preparada para consolidação.


,codigo_cidade,nome_cidade,estado,id_veiculo,placa,ano,modo,origem,km_inicial,capacidade,...,ativo,dt_criacao,criado_id,dt_alteracao,alterado_id,modo_desc,origem_desc,tipo_desc,marca_desc,tipo_combustivel
0,3201159,Brejetuba,Espírito Santo,63203,MSB-5F90,1999,0,2,0.00,24,...,1,NaN,NaN,NaN,NaN,Rodoviário,Veículo Terceirizado,Ônibus,Mercedes-Benz,Não informado
1,1100031,Cabixi,Rondônia,0,OHN-8878,2012,0,1,0.00,14,...,1,NaN,NaN,NaN,NaN,Rodoviário,Veículo Próprio,Micro-ônibus,Iveco,Não informado
2,1100056,Cerejeiras,Rondônia,0,NEG-2868,2014,0,1,0.00,48,...,1,NaN,NaN,NaN,NaN,Rodoviário,Veículo Próprio,Ônibus,Volkswagen,Não informado
3,1100080,Costa Marques,Rondônia,0,NEF-9378,2014,0,1,1839.42,57,...,1,NaN,NaN,NaN,NaN,Rodoviário,Veículo Próprio,Ônibus,Mercedes-Benz,Não informado
4,1100098,Espigão D'Oeste,Rondônia,0,NBW-0502,2013,0,1,0.00,48,...,1,NaN,NaN,NaN,NaN,Rodoviário,Veículo Próprio,Ônibus,Volkswagen,Não informado


In [10]:
# Bloco 3 - Função para consolidar modos por estado

def consolidar_modos_por_estado(df):
    tabela = (
        df
        .groupby("estado")
        .agg(
            total_veiculos=("id_veiculo", "size"),
            rodoviario=("modo_desc", lambda x: (x == "Rodoviário").sum()),
            aquaviario=("modo_desc", lambda x: (x == "Aquaviário").sum()),
            modo_nao_informado=("modo_desc", lambda x: (x == "Não informado").sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de modo por estado:")
    display(tabela)

    return tabela


In [11]:
# Bloco 4 - Consolidar modos

tabela_modos = consolidar_modos_por_estado(df_consolidacao)


Tabela consolidada de modo por estado:


,estado,total_veiculos,rodoviario,aquaviario,modo_nao_informado
0,Acre,133,132,1,0
1,Alagoas,1628,1628,0,0
2,Amapá,57,51,6,0
3,Amazonas,320,202,118,0
4,Bahia,6439,6418,21,0
5,Ceará,3106,3102,4,0
6,Distrito Federal,139,139,0,0
7,Espírito Santo,843,843,0,0
8,Goiás,1858,1854,4,0
9,Maranhão,1753,1750,3,0


In [12]:
# Bloco 5 - Função para consolidar origem por estado

def consolidar_origem_por_estado(df):
    tabela = (
        df
        .groupby("estado")
        .agg(
            total_veiculos=("id_veiculo", "size"),
            proprio=("origem_desc", lambda x: (x == "Veículo Próprio").sum()),
            terceirizado=("origem_desc", lambda x: (x == "Veículo Terceirizado").sum()),
            origem_nao_informada=("origem_desc", lambda x: (x == "Não informado").sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de origem por estado:")
    display(tabela)

    return tabela


tabela_origem = consolidar_origem_por_estado(df_consolidacao)


Tabela consolidada de origem por estado:


,estado,total_veiculos,proprio,terceirizado,origem_nao_informada
0,Acre,133,108,25,0
1,Alagoas,1628,1014,614,0
2,Amapá,57,24,33,0
3,Amazonas,320,135,185,0
4,Bahia,6439,2316,4123,0
5,Ceará,3106,1616,1490,0
6,Distrito Federal,139,138,1,0
7,Espírito Santo,843,253,590,0
8,Goiás,1858,951,907,0
9,Maranhão,1753,593,1160,0


In [13]:
# Bloco 7 - Função para consolidar tipo e combustível por estado

def consolidar_tipo_combustivel_por_estado(df):
    tabela = (
        df
        .groupby("estado")
        .agg(
            total_veiculos=("id_veiculo", "size"),
            onibus=("tipo_desc", lambda x: (x == "Ônibus").sum()),
            micro_onibus=("tipo_desc", lambda x: (x == "Micro-ônibus").sum()),
            van=("tipo_desc", lambda x: (x == "Van").sum()),
            kombi=("tipo_desc", lambda x: (x == "Kombi").sum()),
            caminhão=("tipo_desc", lambda x: (x == "Caminhão").sum()),
            diesel=("tipo_combustivel", lambda x: (x == "Diesel").sum()),
            gasolina=("tipo_combustivel", lambda x: (x == "Gasolina").sum()),
            etanol=("tipo_combustivel", lambda x: (x == "Etanol").sum()),
            gas_natural=("tipo_combustivel", lambda x: (x == "Gás Natural").sum()),
            combustivel_nao_informado=("tipo_combustivel", lambda x: (x == "Não informado").sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de tipo e combustível por estado:")
    display(tabela)

    return tabela


tabela_tipo_comb = consolidar_tipo_combustivel_por_estado(df_consolidacao)


Tabela consolidada de tipo e combustível por estado:


,estado,total_veiculos,onibus,micro_onibus,van,kombi,caminhão,diesel,gasolina,etanol,gas_natural,combustivel_nao_informado
0,Acre,133,71,25,2,0,4,62,6,0,0,65
1,Alagoas,1628,1044,347,179,34,1,1108,28,1,1,490
2,Amapá,57,25,16,0,10,0,44,1,0,0,11
3,Amazonas,320,105,59,5,27,0,187,98,0,0,35
4,Bahia,6439,2723,1237,734,559,4,3093,868,9,0,2448
5,Ceará,3106,1847,874,100,93,14,2199,110,1,0,788
6,Distrito Federal,139,102,37,0,0,0,1,0,0,0,138
7,Espírito Santo,843,350,121,140,212,1,395,176,1,0,271
8,Goiás,1858,720,473,115,471,0,1013,316,22,0,483
9,Maranhão,1753,985,308,105,82,15,1208,163,1,0,376


In [14]:
# Bloco 8 - Consolidar todas as tabelas e exportar para Excel

from functools import reduce


def consolidar_tabelas_por_estado(tabelas):
    tabelas_preparadas = []

    for indice, tabela in enumerate(tabelas):
        tabela_temp = tabela.copy()
        if indice > 0 and "total_veiculos" in tabela_temp.columns:
            tabela_temp = tabela_temp.drop(columns=["total_veiculos"])
        tabelas_preparadas.append(tabela_temp)

    tabela_final = reduce(
        lambda esquerda, direita: esquerda.merge(
            direita,
            on="estado",
            how="outer"
        ),
        tabelas_preparadas
    )

    tabela_final = (
        tabela_final
        .sort_values("estado")
        .reset_index(drop=True)
    )

    linha_brasil = {"estado": "Brasil"}
    for coluna in tabela_final.columns:
        if coluna != "estado":
            linha_brasil[coluna] = tabela_final[coluna].sum()

    tabela_final = pd.concat(
        [
            tabela_final,
            pd.DataFrame([linha_brasil])
        ],
        ignore_index=True
    )

    return tabela_final


tabelas_para_consolidar = [
    tabela_modos,
    tabela_origem,
    tabela_tipo_comb
]

tabela_consolidada_final = consolidar_tabelas_por_estado(tabelas_para_consolidar)

pasta_notebook = obter_pasta_do_notebook()

nome_arquivo_excel = "Consolidação 04 - script_04_veiculos.xlsx"

caminho_arquivo_excel = pasta_notebook / nome_arquivo_excel

tabela_consolidada_final.to_excel(
    caminho_arquivo_excel,
    index=False,
    sheet_name="consolidado"
)

print("Tabela consolidada final exportada com sucesso!")
print(f"Arquivo Excel exportado: {caminho_arquivo_excel}")

display(tabela_consolidada_final)


Tabela consolidada final exportada com sucesso!
Arquivo Excel exportado: c:\Users\Matheus Henrique\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Consolidação 04 - script_04_veiculos.xlsx


,estado,total_veiculos,rodoviario,aquaviario,modo_nao_informado,proprio,terceirizado,origem_nao_informada,onibus,micro_onibus,van,kombi,caminhão,diesel,gasolina,etanol,gas_natural,combustivel_nao_informado
0,Acre,133,132,1,0,108,25,0,71,25,2,0,4,62,6,0,0,65
1,Alagoas,1628,1628,0,0,1014,614,0,1044,347,179,34,1,1108,28,1,1,490
2,Amapá,57,51,6,0,24,33,0,25,16,0,10,0,44,1,0,0,11
3,Amazonas,320,202,118,0,135,185,0,105,59,5,27,0,187,98,0,0,35
4,Bahia,6439,6418,21,0,2316,4123,0,2723,1237,734,559,4,3093,868,9,0,2448
5,Ceará,3106,3102,4,0,1616,1490,0,1847,874,100,93,14,2199,110,1,0,788
6,Distrito Federal,139,139,0,0,138,1,0,102,37,0,0,0,1,0,0,0,138
7,Espírito Santo,843,843,0,0,253,590,0,350,121,140,212,1,395,176,1,0,271
8,Goiás,1858,1854,4,0,951,907,0,720,473,115,471,0,1013,316,22,0,483
9,Maranhão,1753,1750,3,0,593,1160,0,985,308,105,82,15,1208,163,1,0,376
